<a href="https://colab.research.google.com/github/lohithshankarappa/bisLAB/blob/main/Week3AntColony.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import random

# ==========================================
# 1. SETUP THE TRAFFIC NETWORK (GRAPH)
# ==========================================
# Nodes: Intersections (0 to 4)
# Edges: Road segments with (distance, traffic_delay)
# Travel Time = Distance + Traffic Delay
num_nodes = 5
source = 0
destination = 4

# Adjacency matrix for travel times (inf means no direct road)
inf = float('inf')
travel_times = np.array([
    [inf, 10.0, 15.0, inf, inf],  # Node 0 connects to 1 and 2
    [inf, inf, 5.0, 12.0, inf],   # Node 1 connects to 2 and 3
    [inf, inf, inf, 6.0, 20.0],   # Node 2 connects to 3 and 4
    [inf, inf, inf, inf, 8.0],    # Node 3 connects to 4
    [inf, inf, inf, inf, inf]     # Node 4 (Destination)
])

# ==========================================
# 2. ACO PARAMETERS
# ==========================================
num_ants = 10
max_iterations = 50
alpha = 1.0       # Pheromone importance
beta = 2.0        # Heuristic (travel time) importance
rho = 0.1         # Evaporation rate
Q = 100.0         # Pheromone deposit factor
init_pheromone = 1.0

# Initialize pheromone matrix on valid roads
pheromones = np.where(travel_times != inf, init_pheromone, 0.0)

# Visibility heuristic (inverse of travel time: 1 / time)
# Avoid division by zero with a small epsilon or by checking inf
visibility = np.where(travel_times != inf, 1.0 / travel_times, 0.0)

best_route = None
best_time = inf

# ==========================================
# 3. ACO MAIN LOOP
# ==========================================
for iteration in range(max_iterations):
    all_paths = []
    all_times = []

    for ant in range(num_ants):
        current_node = source
        path = [current_node]
        total_time = 0.0
        visited = set([source])

        # Construct route until reaching destination
        while current_node != destination:
            # Find neighboring intersections
            neighbors = [n for n in range(num_nodes) if travel_times[current_node][n] != inf and n not in visited]

            if not neighbors:
                break # Dead end, discard this incomplete path

            # Calculate selection probabilities for neighbors
            probabilities = []
            for n in neighbors:
                tau = pheromones[current_node][n] ** alpha
                eta = visibility[current_node][n] ** beta
                probabilities.append(tau * eta)

            sum_prob = sum(probabilities)
            if sum_prob == 0:
                probabilities = [1.0 / len(neighbors)] * len(neighbors)
            else:
                probabilities = [p / sum_prob for p in probabilities]

            # Probabilistic choice
            next_node = random.choices(neighbors, weights=probabilities, k=1)[0]

            total_time += travel_times[current_node][next_node]
            current_node = next_node
            path.append(current_node)
            visited.add(current_node)

        # Only keep paths that successfully reached the destination
        if current_node == destination:
            all_paths.append(path)
            all_times.append(total_time)

            # Track the global best route
            if total_time < best_time:
                best_time = total_time
                best_route = path

    # ==========================================
    # 4. UPDATE PHEROMONES
    # ==========================================
    # Evaporation step
    pheromones *= (1.0 - rho)

    # Deposit step (only for successful ants)
    for path, t_time in zip(all_paths, all_times):
        for i in range(len(path) - 1):
            u, v = path[i], path[i+1]
            pheromones[u][v] += (Q / t_time)

# ==========================================
# 5. OUTPUT RESULTS
# ==========================================
print(f"Optimization Completed over {max_iterations} iterations.")
print(f"Best Route Found: {' -> '.join(map(str, best_route))}")
print(f"Minimum Travel Time: {best_time:.2f} minutes")


Optimization Completed over 50 iterations.
Best Route Found: 0 -> 2 -> 3 -> 4
Minimum Travel Time: 29.00 minutes
